In [ ]:
# 1. Setup and Authentication
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

auth.authenticate_user()
project_id = 'your-gcp-project-id'
client = bigquery.Client(project=project_id)

# 2. Data Ingestion from BigQuery
# We'll pull the last 24 hours of sensor data
query = """
SELECT 
    timestamp, 
    device_id, 
    temperature, 
    humidity, 
    status 
FROM `your-dataset.iot_telemetry_table`
WHERE timestamp > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
ORDER BY timestamp ASC
"""
df = client.query(query).to_dataframe()

# 3. Exploratory Data Analysis (EDA)
print(df.describe())
print(df['device_id'].value_counts())

# 4. Time-Series Visualization
plt.figure(figsize=(15, 6))
sns.lineplot(data=df, x='timestamp', y='temperature', hue='device_id')
plt.title('Real-time Temperature Monitoring per Device')
plt.xticks(rotation=45)
plt.show()

# 5. Simple Anomaly Detection (The "Smart" part)
# Flagging any temperature 2 standard deviations from the mean
mean_temp = df['temperature'].mean()
std_temp = df['temperature'].std()
df['is_anomaly'] = df['temperature'].apply(lambda x: x > (mean_temp + 2*std_temp))

anomalies = df[df['is_anomaly'] == True]
print(f"Detected {len(anomalies)} anomalies in the last 24 hours.")